# Nvidia "Other" Owner Estimation

This notebook computes the "Other" category of Nvidia chip ownership by subtracting known owners from the total:

**Other = Total Nvidia − Hyperscalers − China − Excluded Other Owners**

The list of excluded other owners (e.g. xAI, Tesla) is configurable via `OTHER_OWNERS_TO_EXCLUDE` in the parameters cell below.

Instead of rerunning the full Nvidia estimation model, it reads confidence intervals from pre-exported CSVs and reconstructs lognormal distributions from the 5th/95th percentiles.

In [51]:
import squigglepy as sq
import numpy as np
import pandas as pd
from squigglepy.numbers import K, M

sq.set_seed(42)
np.random.seed(42)
N_SAMPLES = 5000

# ==============================================
# ADJUSTABLE PARAMETER: which "other" owners to
# subtract before computing the remainder.
# Each name must match an Owner in the input CSVs
# (data_inputs/nvidia_other_owners_totals.csv and
#  data_inputs/nvidia_other_owners_by_chip_type.csv).
# ==============================================
OTHER_OWNERS_TO_EXCLUDE = ['xAI'] # ['xAI', 'Tesla']

CHIP_TYPES = ['A100', 'A800', 'H100/H200', 'H800', 'H20', 'B200', 'B300']
US_CHIPS = ['A100', 'H100/H200', 'B200', 'B300']
CHINA_CHIPS = ['H20', 'H800', 'A800']
HYPERSCALERS = ['Microsoft', 'Meta', 'Amazon', 'Google', 'Oracle']
CHINA_OWNER = 'China'
H100_TOPS = 1979

CHIP_SPECS = {
    'A100':      {'tops': 624,  'tdp': 400},
    'A800':      {'tops': 624,  'tdp': 400},
    'H100/H200': {'tops': 1979, 'tdp': 700},
    'H800':      {'tops': 1979, 'tdp': 700},
    'H20':       {'tops': 296,  'tdp': 400},
    'B200':      {'tops': 5000, 'tdp': 1200},
    'B300':      {'tops': 5000, 'tdp': 1400},
}

## Load CSV exports

Read the pre-computed confidence intervals and reconstruct sample arrays by fitting lognormal distributions to the 5th/95th percentiles via `sq.to()`.

In [52]:
def sample_from_ci(p5, p50, p95, n=N_SAMPLES):
    """Sample from a lognormal fitted to the 5th/95th percentile CI."""
    if p5 <= 0 or p95 <= 0:
        return np.zeros(n)
    return np.array(sq.to(float(p5), float(p95)) @ n)

def end_date_to_quarter(end_date_str):
    """Convert an end date like '6/30/2024' to 'Q2 2024'."""
    dt = pd.to_datetime(end_date_str)
    q = (dt.month - 1) // 3 + 1
    return f'Q{q} {dt.year}'

def quarter_date_strings(cq):
    """Return (start_date, end_date) as M/D/YYYY for a calendar quarter string like 'Q2 2024'."""
    q_num = int(cq[1])
    year = int(cq.split()[-1])
    month_starts = {1: 1, 2: 4, 3: 7, 4: 10}
    month_ends = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}
    start = f"{month_starts[q_num]}/1/{year}"
    end_m, end_d = month_ends[q_num]
    end = f"{end_m}/{end_d}/{year}"
    return start, end

In [53]:
# ==============================================
# LOAD TOTAL NVIDIA CUMULATIVE BY CHIP
# ==============================================
# Each row gives cumulative units AND H100e for one chip type through one quarter.
# We sample both from their respective 5th/95th CIs.

nvidia_by_chip_df = pd.read_csv('csv_export/nvidia_cumulative_by_chip.csv')
nvidia_by_chip_df['quarter'] = nvidia_by_chip_df['End date'].apply(end_date_to_quarter)

# Map CSV chip names to our internal names
CHIP_NAME_MAP = {'H100': 'H100/H200'}

CALENDAR_QUARTERS = list(dict.fromkeys(nvidia_by_chip_df['quarter']))
print(f"Calendar quarters: {CALENDAR_QUARTERS[0]} to {CALENDAR_QUARTERS[-1]} ({len(CALENDAR_QUARTERS)} quarters)")

# Build sample arrays: {quarter: {chip: np.array(N_SAMPLES)}} for both units and H100e
total_running = {cq: {chip: np.zeros(N_SAMPLES) for chip in CHIP_TYPES} for cq in CALENDAR_QUARTERS}
total_running_h100e = {cq: {chip: np.zeros(N_SAMPLES) for chip in CHIP_TYPES} for cq in CALENDAR_QUARTERS}

for _, row in nvidia_by_chip_df.iterrows():
    chip = CHIP_NAME_MAP.get(row['Chip type'], row['Chip type'])
    if chip not in CHIP_TYPES:
        print(f"  Skipping unknown chip type: {row['Chip type']}")
        continue
    cq = row['quarter']
    total_running[cq][chip] = sample_from_ci(
        row['Number of units (5th percentile)'],
        row['Number of units (median)'],
        row['Number of units (95th percentile)'],
    )
    total_running_h100e[cq][chip] = sample_from_ci(
        row['Compute estimate in H100e (5th percentile)'],
        row['Compute estimate in H100e (median)'],
        row['Compute estimate in H100e (95th percentile)'],
    )

# Also load the totals CSV for authoritative total H100e (preserves correlations from original model)
nvidia_totals_df = pd.read_csv('csv_export/nvidia_cumulative_totals.csv')
nvidia_totals_df['quarter'] = nvidia_totals_df['End date'].apply(end_date_to_quarter)

total_h100e = {}
for _, row in nvidia_totals_df.iterrows():
    cq = row['quarter']
    total_h100e[cq] = sample_from_ci(
        row['Compute estimate in H100e (5th percentile)'],
        row['Compute estimate in H100e (median)'],
        row['Compute estimate in H100e (95th percentile)'],
    )

last_cq = CALENDAR_QUARTERS[-1]
print(f"\n{last_cq} total H100e (from totals CSV): {int(np.median(total_h100e[last_cq])):,}")
print(f"{last_cq} total H100e (sum of per-chip): {int(sum(np.median(total_running_h100e[last_cq][c]) for c in CHIP_TYPES)):,}")

Calendar quarters: Q1 2022 to Q1 2026 (17 quarters)

Q1 2026 total H100e (from totals CSV): 14,488,476
Q1 2026 total H100e (sum of per-chip): 14,466,424


In [54]:
# ==============================================
# LOAD HYPERSCALER + CHINA OWNER ESTIMATES
# ==============================================
# Load per-chip unit CIs AND total H100e CIs from the owners CSVs.

owners_by_chip_df = pd.read_csv('owners_csv_export/nvidia_owners_cumulative_by_chip.csv')
owners_by_chip_df['quarter'] = owners_by_chip_df['End date'].apply(end_date_to_quarter)
owners_by_chip_df['chip'] = owners_by_chip_df['Chip type'].map(lambda x: CHIP_NAME_MAP.get(x, x))

owners_totals_df = pd.read_csv('owners_csv_export/nvidia_owners_cumulative_totals.csv')
owners_totals_df['quarter'] = owners_totals_df['End date'].apply(end_date_to_quarter)

KNOWN_OWNERS = HYPERSCALERS + [CHINA_OWNER]

# Per-chip unit samples: {owner: {quarter: {chip: np.array}}}
owner_running = {}
for owner in KNOWN_OWNERS:
    owner_running[owner] = {cq: {chip: np.zeros(N_SAMPLES) for chip in CHIP_TYPES} for cq in CALENDAR_QUARTERS}

for _, row in owners_by_chip_df.iterrows():
    owner = row['Owner']
    if owner not in KNOWN_OWNERS:
        continue
    chip = row['chip']
    if chip not in CHIP_TYPES:
        continue
    cq = row['quarter']
    if cq not in CALENDAR_QUARTERS:
        continue
    owner_running[owner][cq][chip] = sample_from_ci(
        row['Number of Units (5th percentile)'],
        row['Number of Units'],
        row['Number of Units (95th percentile)'],
    )

# Total H100e samples per owner (from the totals CSV, preserving correlations)
owner_h100e = {}
for owner in KNOWN_OWNERS:
    owner_h100e[owner] = {cq: np.zeros(N_SAMPLES) for cq in CALENDAR_QUARTERS}

for _, row in owners_totals_df.iterrows():
    owner = row['Owner']
    if owner not in KNOWN_OWNERS:
        continue
    cq = row['quarter']
    if cq not in CALENDAR_QUARTERS:
        continue
    owner_h100e[owner][cq] = sample_from_ci(
        row['H100e (5th percentile)'],
        row['Compute estimate in H100e (median)'],
        row['H100e (95th percentile)'],
    )

# Quick check
for owner in KNOWN_OWNERS:
    h100e = int(np.median(owner_h100e[owner][last_cq]))
    units = int(sum(np.median(owner_running[owner][last_cq][c]) for c in CHIP_TYPES))
    print(f"{owner} through {last_cq}: {h100e:,} H100e, {units:,} units")

Microsoft through Q1 2026: 3,311,879 H100e, 2,177,757 units
Meta through Q1 2026: 1,950,170 H100e, 1,231,032 units
Amazon through Q1 2026: 1,492,370 H100e, 939,991 units
Google through Q1 2026: 1,295,720 H100e, 844,344 units
Oracle through Q1 2026: 1,096,108 H100e, 628,797 units
China through Q1 2026: 401,675 H100e, 1,799,362 units


## Other owner estimates

For owners in `OTHER_OWNERS_TO_EXCLUDE`:
- Owners with only total H100e data (no chip breakdown) are interpolated to all quarters and split into chip types using the overall non-China chip mix.
- Owners with explicit chip-type breakdowns use those directly.

In [55]:
# ==============================================
# LOAD OTHER OWNER DATA FROM DATA_INPUTS
# ==============================================

other_totals_df = pd.read_csv('data_inputs/nvidia_other_owners_totals.csv')
other_by_chip_df = pd.read_csv('data_inputs/nvidia_other_owners_by_chip_type.csv')

# Parse end dates to calendar quarters
other_totals_df['end_dt'] = pd.to_datetime(other_totals_df['End date'], format='mixed')
other_totals_df['quarter'] = other_totals_df['end_dt'].apply(
    lambda dt: f'Q{(dt.month - 1) // 3 + 1} {dt.year}'
)

other_by_chip_df['end_dt'] = pd.to_datetime(other_by_chip_df['End date'], format='mixed')
other_by_chip_df['quarter'] = other_by_chip_df['end_dt'].apply(
    lambda dt: f'Q{(dt.month - 1) // 3 + 1} {dt.year}'
)
other_by_chip_df['chip'] = other_by_chip_df['Chip type'].map(lambda x: CHIP_NAME_MAP.get(x, x))

# Only include owners that appear in the CSV AND are in our exclude list
available_owners = set(other_totals_df['Owner'].unique()) | set(other_by_chip_df['Owner'].unique())
OTHER_OWNERS = [o for o in OTHER_OWNERS_TO_EXCLUDE if o in available_owners]

missing = set(OTHER_OWNERS_TO_EXCLUDE) - available_owners
if missing:
    print(f"WARNING: owners not found in input CSVs: {missing}")

print(f"Other owners to exclude: {OTHER_OWNERS}")
print(f"\nTotals data:\n{other_totals_df[other_totals_df['Owner'].isin(OTHER_OWNERS)][['Owner', 'quarter', 'Compute estimate in H100e (median)']].to_string()}")
print(f"\nBy-chip data:\n{other_by_chip_df[other_by_chip_df['Owner'].isin(OTHER_OWNERS)][['Owner', 'chip', 'quarter', 'Number of Units']].to_string()}")

Other owners to exclude: ['xAI']

Totals data:
  Owner  quarter  Compute estimate in H100e (median)
8   xAI  Q4 2024                              100000
9   xAI  Q4 2025                              553714

By-chip data:
  Owner       chip  quarter  Number of Units
0   xAI       B200  Q3 2025         140000.0
1   xAI       B200  Q4 2025         140000.0
2   xAI  H100/H200  Q3 2024         100000.0
3   xAI  H100/H200  Q4 2024         100000.0
4   xAI  H100/H200  Q1 2025         200000.0
5   xAI  H100/H200  Q3 2025         200000.0
6   xAI  H100/H200  Q4 2025         200000.0


In [56]:
# ==============================================
# INTERPOLATE TESLA H100e TO ALL QUARTERS, THEN SPLIT BY CHIP
# ==============================================
# Tesla only has total H100e (no chip breakdown), so we:
# 1. Interpolate cumulative H100e to every quarter
# 2. Split into chip types using the non-China chip mix from the total Nvidia model

DEFAULT_UNCERTAINTY = 0

def interpolate_cumulative_h100e(owner_rows, calendar_quarters):
    """Linearly interpolate cumulative H100e for one owner across all quarters.
    
    Returns {quarter: np.array(N_SAMPLES)} sampled with uncertainty.
    """
    cq_to_idx = {cq: i for i, cq in enumerate(calendar_quarters)}
    
    # Collect known data points
    data_points = []
    for _, row in owner_rows.iterrows():
        cq = row['quarter']
        if cq not in cq_to_idx:
            continue
        median = float(row['Compute estimate in H100e (median)'])
        p5 = row.get('H100e (5th percentile)')
        p95 = row.get('H100e (95th percentile)')
        p5 = None if pd.isna(p5) or str(p5).strip() == '' else float(p5)
        p95 = None if pd.isna(p95) or str(p95).strip() == '' else float(p95)
        data_points.append({'idx': cq_to_idx[cq], 'median': median, 'p5': p5, 'p95': p95})
    data_points.sort(key=lambda x: x['idx'])
    
    if not data_points:
        return {cq: np.zeros(N_SAMPLES) for cq in calendar_quarters}
    
    # Interpolate medians
    interpolated = np.zeros(len(calendar_quarters))
    first_idx, last_idx = data_points[0]['idx'], data_points[-1]['idx']
    
    for i in range(len(calendar_quarters)):
        if i < first_idx:
            interpolated[i] = 0.0
        elif i > last_idx:
            interpolated[i] = data_points[-1]['median']
        else:
            prev = next((p for p in reversed(data_points) if p['idx'] <= i), None)
            nxt = next((p for p in data_points if p['idx'] >= i), None)
            if prev['idx'] == nxt['idx']:
                interpolated[i] = prev['median']
            else:
                frac = (i - prev['idx']) / (nxt['idx'] - prev['idx'])
                interpolated[i] = prev['median'] + frac * (nxt['median'] - prev['median'])
    
    # Sample with uncertainty
    result = {}
    for i, cq in enumerate(calendar_quarters):
        med = interpolated[i]
        if med <= 0:
            result[cq] = np.zeros(N_SAMPLES)
            continue
        
        matching = next((p for p in data_points if p['idx'] == i), None)
        if matching and matching['p5'] is not None and matching['p95'] is not None:
            low, high = matching['p5'], matching['p95']
        else:
            low, high = med * (1 - DEFAULT_UNCERTAINTY), med * (1 + DEFAULT_UNCERTAINTY)
        
        result[cq] = np.array(sq.to(max(low, 1), high) @ N_SAMPLES)
    return result


def compute_us_chip_h100e_mix(total_running, calendar_quarters):
    """Fraction of cumulative non-China H100e from each US chip type per quarter.
    
    Returns {quarter: {chip: np.array of fractions}}.
    """
    mix = {}
    for cq in calendar_quarters:
        chip_h100e = {}
        for chip in US_CHIPS:
            if chip in CHIP_SPECS:
                chip_h100e[chip] = total_running[cq][chip] * (CHIP_SPECS[chip]['tops'] / H100_TOPS)
        total_h100e = sum(chip_h100e.values())
        
        mix[cq] = {}
        for chip in US_CHIPS:
            if chip in CHIP_SPECS:
                safe_total = np.where(total_h100e > 0, total_h100e, 1.0)
                mix[cq][chip] = np.where(total_h100e > 0, chip_h100e[chip] / safe_total, 0.0)
    return mix

h100e_chip_mix = compute_us_chip_h100e_mix(total_running, CALENDAR_QUARTERS)

In [57]:
# ==============================================
# BUILD PER-CHIP RUNNING TOTALS FOR EACH OTHER OWNER
# ==============================================
# For each owner in OTHER_OWNERS:
#   - If the owner has per-chip unit data in the by_chip CSV, use that directly
#   - Otherwise, use total H100e from the totals CSV, interpolate, and split by chip mix

other_owner_running = {}
other_owner_cumulative_h100e = {}

for owner in OTHER_OWNERS:
    owner_chip_data = other_by_chip_df[other_by_chip_df['Owner'] == owner]
    owner_total_data = other_totals_df[other_totals_df['Owner'] == owner]
    has_chip_breakdown = len(owner_chip_data) > 0

    if has_chip_breakdown:
        # Owner has explicit per-chip unit counts — interpolate each chip independently
        other_owner_running[owner] = {cq: {chip: np.zeros(N_SAMPLES) for chip in CHIP_TYPES} for cq in CALENDAR_QUARTERS}
        chips_present = owner_chip_data['chip'].unique().tolist()

        for chip in chips_present:
            chip_rows = owner_chip_data[owner_chip_data['chip'] == chip].sort_values('end_dt')
            cq_to_idx = {cq: i for i, cq in enumerate(CALENDAR_QUARTERS)}
            data_points = []
            for _, row in chip_rows.iterrows():
                cq = row['quarter']
                if cq not in cq_to_idx:
                    continue
                units = float(row['Number of Units'])
                p5 = row.get('Number of Units (5th percentile)')
                p95 = row.get('Number of Units (95th percentile)')
                p5 = None if pd.isna(p5) or str(p5).strip() == '' else float(p5)
                p95 = None if pd.isna(p95) or str(p95).strip() == '' else float(p95)
                data_points.append({'idx': cq_to_idx[cq], 'median': units, 'p5': p5, 'p95': p95})
            data_points.sort(key=lambda x: x['idx'])

            if not data_points:
                continue

            interpolated = np.zeros(len(CALENDAR_QUARTERS))
            first_idx, last_idx = data_points[0]['idx'], data_points[-1]['idx']
            for i in range(len(CALENDAR_QUARTERS)):
                if i < first_idx:
                    interpolated[i] = 0.0
                elif i > last_idx:
                    interpolated[i] = data_points[-1]['median']
                else:
                    prev = next((p for p in reversed(data_points) if p['idx'] <= i), None)
                    nxt = next((p for p in data_points if p['idx'] >= i), None)
                    if prev['idx'] == nxt['idx']:
                        interpolated[i] = prev['median']
                    else:
                        frac = (i - prev['idx']) / (nxt['idx'] - prev['idx'])
                        interpolated[i] = prev['median'] + frac * (nxt['median'] - prev['median'])

            for i, cq in enumerate(CALENDAR_QUARTERS):
                med = interpolated[i]
                if med <= 0:
                    continue
                matching = next((p for p in data_points if p['idx'] == i), None)
                if matching and matching['p5'] is not None and matching['p95'] is not None:
                    low, high = matching['p5'], matching['p95']
                else:
                    low, high = med * (1 - DEFAULT_UNCERTAINTY), med * (1 + DEFAULT_UNCERTAINTY)
                other_owner_running[owner][cq][chip] = np.array(sq.to(max(low, 1), high) @ N_SAMPLES)

        # Compute H100e from the per-chip units
        other_owner_cumulative_h100e[owner] = {}
        for cq in CALENDAR_QUARTERS:
            h100e = np.zeros(N_SAMPLES)
            for chip in CHIP_TYPES:
                if chip in CHIP_SPECS:
                    h100e += other_owner_running[owner][cq][chip] * (CHIP_SPECS[chip]['tops'] / H100_TOPS)
            other_owner_cumulative_h100e[owner][cq] = h100e

    else:
        # Owner has only total H100e — interpolate and split by chip mix
        owner_h100e_interp = interpolate_cumulative_h100e(owner_total_data, CALENDAR_QUARTERS)
        other_owner_cumulative_h100e[owner] = owner_h100e_interp

        other_owner_running[owner] = {cq: {} for cq in CALENDAR_QUARTERS}
        for cq in CALENDAR_QUARTERS:
            for chip in CHIP_TYPES:
                if chip in US_CHIPS and chip in CHIP_SPECS:
                    chip_h100e = owner_h100e_interp[cq] * h100e_chip_mix[cq][chip]
                    h100e_mult = CHIP_SPECS[chip]['tops'] / H100_TOPS
                    other_owner_running[owner][cq][chip] = np.where(h100e_mult > 0, chip_h100e / h100e_mult, 0.0)
                else:
                    other_owner_running[owner][cq][chip] = np.zeros(N_SAMPLES)

# Summary
for owner in OTHER_OWNERS:
    h100e_med = int(np.median(other_owner_cumulative_h100e[owner][last_cq]))
    units_med = int(sum(np.median(other_owner_running[owner][last_cq][c]) for c in CHIP_TYPES))
    print(f"{owner} through {last_cq}: {h100e_med:,} H100e, {units_med:,} units")

xAI through Q1 2026: 553,713 H100e, 340,000 units


## Compute remainder

Remainder = Total Nvidia − Hyperscalers − China − Other Owners, computed per-chip to preserve chip-level detail.

In [58]:
# ==============================================
# REMAINDER = TOTAL - HYPERSCALERS - CHINA - TESLA - XAI
# ==============================================
# Use H100e sampled directly from CSVs (not recomputed from units) to preserve
# the correlation structure from the original model.

# Per-chip unit remainder (for chip-level export)
remainder_running = {}
for cq in CALENDAR_QUARTERS:
    remainder_running[cq] = {}
    for chip in CHIP_TYPES:
        chip_total = total_running[cq][chip]
        chip_known = sum(owner_running[owner][cq][chip] for owner in KNOWN_OWNERS)
        chip_others = sum(other_owner_running[owner][cq][chip] for owner in OTHER_OWNERS)
        remainder_running[cq][chip] = chip_total - chip_known - chip_others

# H100e remainder using the authoritative totals from CSVs
remainder_h100e = {}
for cq in CALENDAR_QUARTERS:
    known_h100e = sum(owner_h100e[owner][cq] for owner in KNOWN_OWNERS)
    other_h100e = sum(other_owner_cumulative_h100e[owner][cq] for owner in OTHER_OWNERS)
    remainder_h100e[cq] = total_h100e[cq] - known_h100e - other_h100e

# Summary
all_owner_labels = HYPERSCALERS + [CHINA_OWNER] + OTHER_OWNERS + ['Remainder']

print(f"Cumulative H100e through {last_cq}:")
print(f"  {'Total Nvidia':<20}{int(np.median(total_h100e[last_cq])):>12,}")
for owner in KNOWN_OWNERS:
    print(f"  {owner:<20}{int(np.median(owner_h100e[owner][last_cq])):>12,}")
for owner in OTHER_OWNERS:
    print(f"  {owner:<20}{int(np.median(other_owner_cumulative_h100e[owner][last_cq])):>12,}")
print(f"  {'Remainder':<20}{int(np.median(remainder_h100e[last_cq])):>12,}")

Cumulative H100e through Q1 2026:
  Total Nvidia          14,488,476
  Microsoft              3,311,879
  Meta                   1,950,170
  Amazon                 1,492,370
  Google                 1,295,720
  Oracle                 1,096,108
  China                    401,675
  xAI                      553,713
  Remainder              4,267,571


In [59]:
# ==============================================
# FULL SUMMARY TABLE: H100e BY OWNER OVER TIME
# ==============================================

owner_h100e_series = {}
for owner in KNOWN_OWNERS:
    owner_h100e_series[owner] = owner_h100e[owner]
for owner in OTHER_OWNERS:
    owner_h100e_series[owner] = other_owner_cumulative_h100e[owner]
owner_h100e_series['Remainder'] = remainder_h100e

rows = []
for cq in CALENDAR_QUARTERS:
    row = {'Quarter': cq}
    row['Total'] = int(np.median(total_h100e[cq]))
    for label in all_owner_labels:
        row[label] = int(np.median(owner_h100e_series[label][cq]))
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Quarter')
print("Cumulative H100e by owner (median):\n")
print(summary_df.to_string())

Cumulative H100e by owner (median):

            Total  Microsoft     Meta   Amazon   Google   Oracle   China     xAI  Remainder
Quarter                                                                                    
Q1 2022     67016      14775     7848     5664     5705     1606    9491       0      20222
Q2 2022    138946      30463    16245    11827    11640     3415   19615       0      42110
Q3 2022    212983      47476    24660    18378    17941     5160   22832       0      71553
Q4 2022    290639      64944    34371    25156    24884     7850   32543       0      95715
Q1 2023    382417      85646    45266    32825    32703    11237   48987       0     120782
Q2 2023    594503     133569    69694    50178    50635    19188   89601       0     175466
Q3 2023    932346     207186   110056    79282    79267    31717  154715       0     261930
Q4 2023   1385003     309683   161832   117208   116916    54707  177711       0     437075
Q1 2024   1949473     443719   232995   169

## Export CSVs

Export the "Other" remainder to `owners_csv_export/` in the same format as the original notebook.

In [60]:
# ==============================================
# EXPORT TO CSV
# ==============================================

from datetime import datetime as _dt

timestamp = _dt.now().strftime("%m-%d-%Y %H:%M")
output_dir = 'owners_csv_export'

# Determine incomplete quarters from the Nvidia totals CSV
# Quarters marked "checked" in the Incomplete column are incomplete
incomplete_quarters = set()
for _, row in nvidia_totals_df.iterrows():
    if str(row.get('Incomplete', '')).strip().lower() == 'checked':
        incomplete_quarters.add(row['quarter'])

named_owners = HYPERSCALERS + [CHINA_OWNER] + OTHER_OWNERS
named_owners_str = ', '.join(named_owners)

def tail_cols(cq=None):
    base_note = f"Excludes {named_owners_str}. Estimates generated on: {timestamp}"
    if cq in incomplete_quarters:
        base_note = f"Excludes {named_owners_str}. Incomplete: based on Nvidia fiscal quarters ending 1/25/2026. Estimates generated on: {timestamp}"
    return {
        'Source / Link': '',
        'Notes': base_note,
    }

OWNER_LABEL = 'Other'
# Only export non-China chip types (China chips are allocated entirely to the China owner)
EXPORT_CHIPS = [c for c in CHIP_TYPES if c not in CHINA_CHIPS]
first_q_start, _ = quarter_date_strings(CALENDAR_QUARTERS[0])

# --- 1. Cumulative totals CSV ---
# Use remainder_h100e (sampled from authoritative CSV totals) for H100e columns,
# and per-chip unit remainder summed for the units columns.
cumulative_rows = []
for cq in CALENDAR_QUARTERS:
    unit_samples = sum(remainder_running[cq][chip] for chip in EXPORT_CHIPS)
    h100e_samples = remainder_h100e[cq]
    power_w = sum(
        remainder_running[cq][chip] * CHIP_SPECS[chip]['tdp']
        for chip in EXPORT_CHIPS if chip in CHIP_SPECS
    )
    power_mw = power_w / 1e6

    _, end_date = quarter_date_strings(cq)
    row = {
        'Name': f"Other cumulative Nvidia through {cq}",
        'Chip manufacturer': 'Nvidia',
        'Owner': OWNER_LABEL,
        'Start date': first_q_start,
        'End date': end_date,
        'Compute estimate in H100e (median)': int(np.percentile(h100e_samples, 50)),
        'H100e (5th percentile)': int(np.percentile(h100e_samples, 5)),
        'H100e (95th percentile)': int(np.percentile(h100e_samples, 95)),
        'Number of Units': int(np.percentile(unit_samples, 50)),
        'Number of Units (5th percentile)': int(np.percentile(unit_samples, 5)),
        'Number of Units (95th percentile)': int(np.percentile(unit_samples, 95)),
        'Total TDP (W)': int(np.percentile(power_w, 50)),
        'Total TDP (W) (5th percentile)': int(np.percentile(power_w, 5)),
        'Total TDP (W) (95th percentile)': int(np.percentile(power_w, 95)),
        'Power in MW (median)': round(np.percentile(power_mw, 50), 2),
        'Power in MW (5th percentile)': round(np.percentile(power_mw, 5), 2),
        'Power in MW (95th percentile)': round(np.percentile(power_mw, 95), 2),
        'Incomplete': 'checked' if cq in incomplete_quarters else '',
    }
    row.update(tail_cols(cq))
    cumulative_rows.append(row)

cumulative_df = pd.DataFrame(cumulative_rows)
cumulative_df.to_csv(f'{output_dir}/nvidia_owners_other_cumulative_totals.csv', index=False)
print(f"Exported {len(cumulative_df)} rows to {output_dir}/nvidia_owners_other_cumulative_totals.csv")

# --- 2. Cumulative by chip type CSV ---
# Derive per-chip H100e from unit remainder, then scale so that chip H100e medians
# sum to the authoritative totals H100e median. This anchors the per-chip breakdown
# to the totals while preserving the relative chip mix from the unit model.
by_chip_rows = []
for cq in CALENDAR_QUARTERS:
    # First compute unscaled per-chip H100e from unit remainder
    chip_h100e_raw = {}
    for chip in EXPORT_CHIPS:
        if chip not in CHIP_SPECS:
            continue
        unit_samples = remainder_running[cq][chip]
        if np.median(unit_samples) <= 0:
            continue
        chip_h100e_raw[chip] = unit_samples * (CHIP_SPECS[chip]['tops'] / H100_TOPS)

    # Scale factor: ratio of totals H100e median to sum-of-chips H100e median
    if chip_h100e_raw:
        raw_sum_median = sum(np.median(v) for v in chip_h100e_raw.values())
        total_median = np.median(remainder_h100e[cq])
        scale = total_median / raw_sum_median if raw_sum_median > 0 else 1.0
    else:
        scale = 1.0

    for chip in EXPORT_CHIPS:
        if chip not in chip_h100e_raw:
            continue
        unit_samples = remainder_running[cq][chip]
        h100e_samples = chip_h100e_raw[chip] * scale
        tdp_w = unit_samples * CHIP_SPECS[chip]['tdp']

        _, end_date = quarter_date_strings(cq)
        row = {
            'Name': f"Other {chip} cumulative through {cq}",
            'Chip manufacturer': 'Nvidia',
            'Owner': OWNER_LABEL,
            'Start date': first_q_start,
            'End date': end_date,
            'Compute estimate in H100e (median)': int(np.percentile(h100e_samples, 50)),
            'H100e (5th percentile)': int(np.percentile(h100e_samples, 5)),
            'H100e (95th percentile)': int(np.percentile(h100e_samples, 95)),
            'Number of Units': int(np.percentile(unit_samples, 50)),
            'Number of Units (5th percentile)': int(np.percentile(unit_samples, 5)),
            'Number of Units (95th percentile)': int(np.percentile(unit_samples, 95)),
            'Total TDP (W)': int(np.percentile(tdp_w, 50)),
            'Total TDP (W) (5th percentile)': int(np.percentile(tdp_w, 5)),
            'Total TDP (W) (95th percentile)': int(np.percentile(tdp_w, 95)),
            'Chip type': chip,
            'Incomplete': 'checked' if cq in incomplete_quarters else '',
        }
        row.update(tail_cols(cq))
        by_chip_rows.append(row)

by_chip_df = pd.DataFrame(by_chip_rows)
by_chip_df.to_csv(f'{output_dir}/nvidia_owners_other_cumulative_by_chip.csv', index=False)
print(f"Exported {len(by_chip_df)} rows to {output_dir}/nvidia_owners_other_cumulative_by_chip.csv")

# Sanity check: compare by-chip H100e sum vs totals H100e for last quarter
last_end = quarter_date_strings(CALENDAR_QUARTERS[-1])[1]
last_chips = by_chip_df[by_chip_df['End date'] == last_end]
chip_h100e_sum = last_chips['Compute estimate in H100e (median)'].sum()
total_h100e_med = cumulative_df.iloc[-1]['Compute estimate in H100e (median)']
print(f"\n{CALENDAR_QUARTERS[-1]} sanity check:")
print(f"  By-chip H100e sum: {int(chip_h100e_sum):,}")
print(f"  Totals H100e:      {int(total_h100e_med):,}")
print(f"  Difference:        {int(chip_h100e_sum - total_h100e_med):,}")

# --- 3. Quarterly (non-cumulative) by chip type CSV ---
# Load quarterly totals and known-owner quarterly data from upstream CSVs,
# then compute: Other quarterly = Total quarterly - known owners quarterly - other owners quarterly

# Total Nvidia quarterly by chip
total_q_df = pd.read_csv('csv_export/nvidia_calendar_quarter_chip_timelines.csv')
total_q_df['quarter'] = total_q_df['End date'].apply(end_date_to_quarter)
total_q_df['chip'] = total_q_df['Chip type'].map(lambda x: CHIP_NAME_MAP.get(x, x))

# Sample total quarterly units per chip
total_quarterly = {}
for _, row in total_q_df.iterrows():
    cq, chip = row['quarter'], row['chip']
    if chip not in CHIP_TYPES or cq not in CALENDAR_QUARTERS:
        continue
    total_quarterly.setdefault(cq, {})[chip] = sample_from_ci(
        row['Number of Units (5th percentile)'],
        row['Number of Units'],
        row['Number of Units (95th percentile)'],
    )

# Known-owner (hyperscalers + China) quarterly by chip, ignoring "Other" rows
owners_q_df = pd.read_csv('owners_csv_export/nvidia_owners_quarters_by_chip.csv')
owners_q_df['quarter'] = owners_q_df['End date'].apply(end_date_to_quarter)
owners_q_df['chip'] = owners_q_df['Chip type'].map(lambda x: CHIP_NAME_MAP.get(x, x))

known_quarterly = {}
for _, row in owners_q_df.iterrows():
    owner = row['Owner']
    if owner not in KNOWN_OWNERS:
        continue
    cq, chip = row['quarter'], row['chip']
    if chip not in CHIP_TYPES or cq not in CALENDAR_QUARTERS:
        continue
    known_quarterly.setdefault(cq, {}).setdefault(chip, np.zeros(N_SAMPLES))
    known_quarterly[cq][chip] += sample_from_ci(
        row['Number of Units (5th percentile)'],
        row['Number of Units'],
        row['Number of Units (95th percentile)'],
    )

# Other-owner quarterly by chip: diff consecutive cumulative per-owner samples
other_owner_quarterly = {}
for owner in OTHER_OWNERS:
    for i, cq in enumerate(CALENDAR_QUARTERS):
        for chip in CHIP_TYPES:
            now = other_owner_running[owner][cq][chip]
            prev = other_owner_running[owner][CALENDAR_QUARTERS[i - 1]][chip] if i > 0 else np.zeros(N_SAMPLES)
            q_units = now - prev
            if np.median(q_units) <= 0:
                continue
            other_owner_quarterly.setdefault(cq, {}).setdefault(chip, np.zeros(N_SAMPLES))
            other_owner_quarterly[cq][chip] += q_units

# Remainder = total - known owners - other owners, per quarter per chip
quarterly_by_chip_rows = []
for cq in CALENDAR_QUARTERS:
    if cq not in total_quarterly:
        continue
    start_date, end_date = quarter_date_strings(cq)
    for chip in EXPORT_CHIPS:
        if chip not in CHIP_SPECS:
            continue
        total_q = total_quarterly.get(cq, {}).get(chip, np.zeros(N_SAMPLES))
        known_q = known_quarterly.get(cq, {}).get(chip, np.zeros(N_SAMPLES))
        other_q = other_owner_quarterly.get(cq, {}).get(chip, np.zeros(N_SAMPLES))
        remainder_q = total_q - known_q - other_q

        if np.median(remainder_q) <= 0:
            continue

        h100e_q = remainder_q * (CHIP_SPECS[chip]['tops'] / H100_TOPS)
        tdp_q = remainder_q * CHIP_SPECS[chip]['tdp']

        row = {
            'Name': f"Other {chip} {cq}",
            'Chip manufacturer': 'Nvidia',
            'Owner': OWNER_LABEL,
            'Start date': start_date,
            'End date': end_date,
            'Compute estimate in H100e (median)': int(np.percentile(h100e_q, 50)),
            'H100e (5th percentile)': int(np.percentile(h100e_q, 5)),
            'H100e (95th percentile)': int(np.percentile(h100e_q, 95)),
            'Number of Units': int(np.percentile(remainder_q, 50)),
            'Number of Units (5th percentile)': int(np.percentile(remainder_q, 5)),
            'Number of Units (95th percentile)': int(np.percentile(remainder_q, 95)),
            'Total TDP (W)': int(np.percentile(tdp_q, 50)),
            'Total TDP (W) (5th percentile)': int(np.percentile(tdp_q, 5)),
            'Total TDP (W) (95th percentile)': int(np.percentile(tdp_q, 95)),
            'Chip type': chip,
            'Incomplete': 'checked' if cq in incomplete_quarters else '',
        }
        row.update(tail_cols(cq))
        quarterly_by_chip_rows.append(row)

quarterly_by_chip_df = pd.DataFrame(quarterly_by_chip_rows)
quarterly_by_chip_df.to_csv(f'{output_dir}/nvidia_owners_other_quarters_by_chip.csv', index=False)
print(f"Exported {len(quarterly_by_chip_df)} rows to {output_dir}/nvidia_owners_other_quarters_by_chip.csv")

Exported 17 rows to owners_csv_export/nvidia_owners_other_cumulative_totals.csv
Exported 42 rows to owners_csv_export/nvidia_owners_other_cumulative_by_chip.csv

Q1 2026 sanity check:
  By-chip H100e sum: 4,267,569
  Totals H100e:      4,267,571
  Difference:        -2
Exported 30 rows to owners_csv_export/nvidia_owners_other_quarters_by_chip.csv
